<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 13. GAN — Generative Adversarial Nets

- **원 논문:** [Generative Adversarial Nets](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)
- **저자 / 발표:** Ian J. Goodfellow et al. · NeurIPS (2014)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, binary cross entropy, 확률분포, 역전파

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

2차원 Gaussian 데이터에 대해 MLP 생성자와 판별자를 번갈아 갱신하고, 논문이 제안한 non-saturating 생성자 목적함수로 고정 noise 표본의 분포 오차가 줄어드는지 확인한다.

**원 논문과 다른 것**

논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3, Eq. (1) | `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수 | 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사 |
| §3, Algorithm 1 및 Eq. (1) 직후 문단 | `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신 | 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사 |
| §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6) | `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단 | 학습 전후 MMD와 판별자 평균 출력을 비교 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\min_G\max_D\;
\mathbb{E}_{x\sim p_{data}}[\log D(x)]
+\mathbb{E}_{z\sim p(z)}[\log(1-D(G(z)))]
$$

- generator $G$는 noise를 sample로, discriminator $D$는 sample을 real 확률로 변환합니다.
- 두 모델의 목표가 반대인 minimax game이므로 optimizer와 gradient 경로를 분리합니다.
- discriminator step과 generator step을 독립 함수로 구현하고 해당 step의 parameter만 변하는지 보장합니다.
- $z:[B,D_z]\rightarrow G(z):[B,D_x]\rightarrow D(\cdot):[B,1]$의 흐름입니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3, Eq. (1)
- **노트북 구현:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문의 게임을 2차원으로 축소하기

논문 §3 Eq. (1)은

$$\min_G\max_D\;\mathbb E_{x\sim p_{data}}\log D(x)
+\mathbb E_{z\sim p_z}\log(1-D(G(z)))$$

를 정의합니다. 여기서는 타원형 2차원 Gaussian을 실제 분포로 두고 두 MLP를
번갈아 갱신합니다. 생성자에는 같은 절에서 권한 `maximize log D(G(z))`, 즉
non-saturating loss를 사용합니다. 이 변형은 고정점은 같고 초기 gradient가 더 강합니다.

**성공 기준:** 고정 noise에서 생성 분포와 실제 분포 사이 RBF-MMD가 초기보다 작아야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(130)
np.random.seed(130)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This tiny teaching batch may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger variants.

real_mean = torch.tensor([1.4, -0.8])
real_transform = torch.tensor([[0.75, 0.0], [0.28, 0.42]])


def sample_real(n, generator):
    base = torch.randn(n, 2, generator=generator)
    return (real_mean + base @ real_transform.T).to(DEVICE)


def sample_noise(n, generator):
    return torch.randn(n, 2, generator=generator).to(DEVICE)


data_rng = torch.Generator().manual_seed(131)
real_data = sample_real(1024, data_rng)
print(ACCELERATOR.summary())
print(f"real mean={real_data.mean(0).tolist()}, shape={tuple(real_data.shape)}")
assert real_data.shape == (1024, 2)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Eq. (1)
- **이 셀의 책임:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 2 -> 32 -> 32 -> 2 MLP를 정의하세요.
# TODO: 2 -> 32 -> 32 -> 1 logit MLP를 정의하세요.
class Generator(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, z):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class Discriminator(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def discriminator_loss(real_logits, fake_logits):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def generator_loss(fake_logits):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


G, D = ACCELERATOR.move(Generator(), Discriminator())
assert G(torch.zeros(5, 2, device=DEVICE)).shape == (5, 2)
assert D(torch.zeros(5, 2, device=DEVICE)).shape == (5,)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 1: fake를 detach하고 D의 실제/가짜 BCE update를 수행하세요.
# TODO 2: 새 noise를 뽑아 non-saturating G update를 수행하세요.
def gan_step(G, D, g_optimizer, d_optimizer, batch_size, generator):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 고정 probe noise의 초기 출력을 저장하고 600번 교대 update를 수행하세요.
train_rng = torch.Generator().manual_seed(132)
probe_rng = torch.Generator().manual_seed(133)
probe_noise = sample_noise(384, probe_rng)
initial_fake = None
history = []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6)
- **이 셀의 책임:** `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 학습 전후 MMD와 판별자 평균 출력을 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: RBF-MMD를 구현해 initial/final 분포 오차를 비교하고 산점도와 loss를 그리세요.
def rbf_mmd(x, y, sigma=1.0):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def kernel(a, b):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 7. 해석 TODO

1. GAN loss가 단조감소하지 않아도 학습이 실패했다고 단정할 수 없는 이유를 쓰세요.
2. `fake_batch.detach()`가 막는 gradient 경로를 설명하세요.
3. Proposition 1의 이상적 균형에서 `D(real)`과 `D(fake)`는 어디로 가야 할까요?

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.